In [1]:
# import sys
# !{sys.executable} -m ensurepip --upgrade

In [2]:
# import sys
# !{sys.executable} -m pip install pygris

In [3]:
# pip install census

In [4]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import google.auth
import os
import gcsfs
import requests
from pygris import tracts 
from pygris.utils import erase_water
fs = gcsfs.GCSFileSystem()
pd.set_option('display.max_columns', None)
from pygris import block_groups

In [5]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships'

In [6]:
with open ("ACS_apikey", "r") as file:
    api_key = file.read().strip()

In [7]:
variables = [
    "B01003_001E",  # total population
    "B19013_001E",  # median household income
    "B23025_003E",  # employed population
    "B25044_003E",  # owner-occupied households with no vehicle
    "B25044_010E",  # renter-occupied households with no vehicle
    # youth by age/sex
    "B01001_003E", "B01001_004E", "B01001_005E", "B01001_006E",
    "B01001_027E", "B01001_028E", "B01001_029E", "B01001_030E",
    # income brackets
    "B19001_002E", "B19001_003E", "B19001_004E", "B19001_005E",
    "B19001_006E", "B19001_007E", "B19001_008E", "B19001_009E",
    "B19001_010E", "B19001_011E", "B19001_012E",
]

In [8]:
# variables = [
#     "B01003_001E",  # total population
#     "B17001_002E",  # poverty count
#     "B19013_001E" , # median household income
#     "B08201_002E",  # households with no vehicle
#     "B23025_003E",  # employed population
#     "B21001_002E",  # veterans
#     "B18101_001E"  # total disabled population universe
#     ]

In [9]:
def chunk(lst, size=20):
    return [lst[i:i+size] for i in range(0, len(lst), size)]

In [10]:
def fetch_acs(vars_subset, api_key):

    var_str = "NAME," + ",".join(vars_subset)

    url = (
        "https://api.census.gov/data/2024/acs/acs5"
        f"?get={var_str}"
        "&for=block%20group:*"
        "&in=state:06"
        "&in=county:*"
        "&in=tract:*"
        f"&key={api_key}"
    )

    r = requests.get(url)

    if r.status_code != 200:
        print("Error:", r.text)
        return None

    data = r.json()
    df = pd.DataFrame(data[1:], columns=data[0])

    # GEOID (12-digit block group)
    df["GEOID"] = (
        df["state"] + df["county"] + df["tract"] + df["block group"]
    )

    return df

In [11]:
chunks = chunk(variables, size=20)

dfs = [fetch_acs(c, api_key) for c in chunks]
dfs = [df for df in dfs if df is not None]


# -----------------------------
# 5. Combine columns correctly (NO MERGE)
# -----------------------------
census_data = dfs[0].copy()

for df in dfs[1:]:
    df = df.drop(columns=["NAME", "state", "county", "tract", "block group"], errors="ignore")
    census_data = pd.concat([census_data, df], axis=1)

# remove duplicate columns if any
census_data = census_data.loc[:, ~census_data.columns.duplicated()]


In [12]:
census_data["county_name"] = census_data["NAME"].str.extract(
    r';\s*(.*?)(?: County)?;'
)

census_data = census_data.drop(columns=["NAME"])


In [13]:
num_cols = census_data.columns.difference(["GEOID", "county_name"])
census_data[num_cols] = census_data[num_cols].astype(int)


In [14]:
census_data = census_data.rename(columns={
    "B01003_001E": "total_pop",
    "B19013_001E": "median_household_income",
    "B23025_003E": "employed_pop",
    "B25044_003E": "owner_occ_no_vehicle",
    "B25044_010E": "renter_occ_no_vehicle",
    # youth by age/sex
    "B01001_003E": "male_under5",
    "B01001_004E": "male_5_9",
    "B01001_005E": "male_10_14",
    "B01001_006E": "male_15_17",
    "B01001_027E": "female_under5",
    "B01001_028E": "female_5_9",
    "B01001_029E": "female_10_14",
    "B01001_030E": "female_15_17",
    # income brackets
    "B19001_002E": "income_less_10000",
    "B19001_003E": "income_10000_14999",
    "B19001_004E": "income_15000_19999",
    "B19001_005E": "income_20000_24999",
    "B19001_006E": "income_25000_29999",
    "B19001_007E": "income_30000_34999",
    "B19001_008E": "income_35000_39999",
    "B19001_009E": "income_40000_44999",
    "B19001_010E": "income_45000_49999",
    "B19001_011E": "income_50000_59999",
    "B19001_012E": "income_60000_74999",
})


In [15]:
exclude = ['state', 'county', 'block group', 'county_name', 'GEOID']
cols_to_numeric = [col for col in census_data.columns if col not in exclude]
census_data[cols_to_numeric] = census_data[cols_to_numeric].apply(pd.to_numeric, errors='coerce')

In [16]:
census_data['households_no_vehicle'] = (
    census_data['owner_occ_no_vehicle'] + 
    census_data['renter_occ_no_vehicle']
)

census_data['total_youth'] = (
    census_data['male_under5'] + census_data['male_5_9'] +
    census_data['male_10_14'] + census_data['male_15_17'] +
    census_data['female_under5'] + census_data['female_5_9'] +
    census_data['female_10_14'] + census_data['female_15_17']
)

census_data['inc_extremelylow'] = (
    census_data['income_less_10000'] + census_data['income_10000_14999'] +
    census_data['income_15000_19999'] + census_data['income_20000_24999']
)
census_data['inc_verylow'] = (
    census_data['income_25000_29999'] + census_data['income_30000_34999'] +
    census_data['income_35000_39999'] + census_data['income_40000_44999'] +
    census_data['income_45000_49999']
)
census_data['inc_low'] = (
    census_data['income_50000_59999'] + 
    census_data['income_60000_74999']
)
census_data['inc_total_lowincome'] = (
    census_data['inc_extremelylow'] + 
    census_data['inc_verylow'] + 
    census_data['inc_low']
)

In [17]:
census_data = census_data[['state', 'county', 'block group', 'county_name', 'GEOID', 'total_pop', 'median_household_income', 'employed_pop', 'households_no_vehicle', 
                           'total_youth', 'inc_extremelylow', 'inc_verylow', 'inc_low', 'inc_total_lowincome', ]]

In [18]:
census_data.head(2)

,state,county,block group,county_name,GEOID,total_pop,median_household_income,employed_pop,households_no_vehicle,total_youth,inc_extremelylow,inc_verylow,inc_low,inc_total_lowincome
0,6,1,1,Census Tract 4001,060014001001,2074,250001,1202,57,428,25,46,9,80
1,6,1,2,Census Tract 4001,060014001002,1058,209286,582,50,157,58,0,35,93


In [19]:
# Retrieving Block Group Geometries for California
ca_bg_full = block_groups(state="CA", cb=True, year=2024, cache=True)
ca_bg_full.to_crs(3310, inplace=True)
ca_bg_full = ca_bg_full.explode(index_parts=False).reset_index(drop=True)
ca_bg_full = ca_bg_full[~ca_bg_full.is_empty]

Using FIPS code '06' for input 'CA'


In [20]:
ca_block_land = erase_water(ca_bg_full.copy())

/home/jovyan/data-analyses/.venv/lib/python3.11/site-packages/geopandas/geodataframe.py:2475: UserWarning: `keep_geom_type=True` in overlay resulted in 666 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  return geopandas.overlay(


In [21]:
# Merging the census tract geometries with the census data based on the GEOID
block_ca_acs = ca_block_land.merge(census_data, how="inner", on="GEOID")

In [22]:
# Calculate the area of each census tract in square meters.
block_ca_acs["area_m2"] = block_ca_acs.geometry.area

In [23]:
# Store data in warehouse
with fs.open(f"{GCS_FILE_PATH}/AHSC_2026/census_blocks_data.parquet", "wb") as f:
    block_ca_acs.to_parquet(f, index=False)